# ETL — Sistema de Monitoreo Demográfico Comunal de Aysén

Este notebook implementa el proceso de Extracción, Transformación y Carga (ETL) de los datos del CENSO INE 2017 y 2024 para la Región de Aysén.

**Estructura del proceso:**
1. Extracción de datos crudos (CSV del INE)
2. Transformación: filtro por región, limpieza, normalización
3. Cálculo de indicadores demográficos
4. Exportación de archivos procesados (para carga posterior en PostgreSQL)

---

## 0. Configuración e Imports

In [1]:
import pandas as pd

# ── Rutas de archivos ──────────────────────────────────────────────────────────
RUTA_CENSO_2017 = 'data/personas_censo2017.csv'
RUTA_CENSO_2024 = 'data/personas_censo2024.csv'

RUTA_SALIDA_2024 = 'data/censo_2024_aysen_final.csv'

# ── Constantes del dominio ─────────────────────────────────────────────────────
REGION_AYSEN = 11

COMUNAS = {
    11101: 'Coyhaique',
    11102: 'Lago Verde',
    11201: 'Aysén',
    11202: 'Cisnes',
    11203: 'Guaitecas',
    11301: 'Cochrane',
    11302: "O'Higgins",
    11303: 'Tortel',
    11401: 'Chile Chico',
    11402: 'Río Ibáñez'
}

COLUMNAS_2024 = ['region', 'comuna', 'sexo', 'edad', 'edad_quinquenal']

print("Configuración lista.")

Configuración lista.


---
## 1. EXTRACCIÓN

### 1.1 Extracción CENSO 2017

El archivo del 2017 ya es específico de Aysén, por lo que se carga directamente.

In [2]:
df_aysen_2017 = pd.read_csv(RUTA_CENSO_2017, sep=',')

print(f"Registros CENSO 2017 cargados: {len(df_aysen_2017):,}")
print(f"Columnas: {df_aysen_2017.columns.tolist()}")
df_aysen_2017.head(3)

Registros CENSO 2017 cargados: 607
Columnas: ['REGION', 'PROVINCIA', 'NOMBRE COMUNA', 'DISTRITO', 'LOCALIDAD', 'ENTIDAD', 'CATEGORIA', 'Número total de personas', 'Total de Hombres', 'Total de Mujeres', 'Total de personas de 0 a 5 años', 'Total de personas de 6 a 14 años', 'Total de personas de 15 a 64 años', 'Total de personas de 65 y más años', 'Total personas migrantes que residen habitualmente en el territorio nacional', 'Total personas que se consideran pertenecientes a algún pueblo indígena u originario', 'Total de viviendas particulares', 'Total viviendas colectivas', 'Total viviendas ocupadas con moradores presentes', 'Total viviendas', 'Cantidad de hogares', 'Cantidad de viviendas tipo casa', 'Cantidad de viviendas tipo departamento en edificio', 'Cantidad de viviendas tipo vivienda tradicional indígena (ruka, pae pae u otras)', 'Cantidad de viviendas tipo pieza en casa antigua o en conventillo', 'Cantidad de viviendas tipo mediagua, mejora, rancho o choza', 'Cantidad de vivie

,REGION,PROVINCIA,NOMBRE COMUNA,DISTRITO,LOCALIDAD,ENTIDAD,CATEGORIA,Número total de personas,Total de Hombres,Total de Mujeres,...,Viviendas con materialidad de piso tierra,Total viviendas con materialidad aceptable,Total viviendas con materialidad recuperable,Total viviendas con materialidad irrecuperable,Cantidad de viviendas con origen del agua por red pública,Cantidad de viviendas con origen del agua por pozo o noria,Cantidad de viviendas con origen del agua por camión aljibe,"Cantidad de viviendas con origen del agua por río, vertiente, estero, canal, lago, etc.",x,y
0,REGIÓN DE AYSÉN DEL GENERAL CARLOS IBÁÑEZ DEL ...,COYHAIQUE,COYHAIQUE,7,ENSENADA VALLE SIMPSON,ENSENADA VALLE SIMPSON,Parcela-Hijuela,178,94,84,...,0,50,23,4,0,47,6,23,NaN,NaN
1,REGIÓN DE AYSÉN DEL GENERAL CARLOS IBÁÑEZ DEL ...,COYHAIQUE,COYHAIQUE,7,LA CORDONADA,LA CORDONADA PONIENTE,Parcela-Hijuela,15,9,6,...,0,6,1,0,2,3,0,2,NaN,NaN
2,REGIÓN DE AYSÉN DEL GENERAL CARLOS IBÁÑEZ DEL ...,COYHAIQUE,COYHAIQUE,7,VALLE SIMPSON,CALLEJÓN MOISÉS MANRÍQUEZ,Parcela-Hijuela,22,16,6,...,0,8,3,0,0,4,0,7,NaN,NaN


### 1.2 Extracción CENSO 2024 (procesamiento por chunks)

El archivo 2024 contiene datos de todo Chile. Se procesa en chunks para no saturar la memoria RAM y se filtra directamente por región 11.

In [3]:
lista_aysen = []
lector = pd.read_csv(
    RUTA_CENSO_2024,
    sep=';',
    usecols=COLUMNAS_2024,
    chunksize=100_000,
    low_memory=False
)

for i, chunk in enumerate(lector):
    aysen_chunk = chunk[chunk['region'] == REGION_AYSEN]
    if not aysen_chunk.empty:
        lista_aysen.append(aysen_chunk)
    if i % 10 == 0:
        print(f"  Procesando chunk {i}...")

df_aysen_2024 = pd.concat(lista_aysen, ignore_index=True)

print(f"\n✓ Extracción completa.")
print(f"  Registros de Aysén encontrados: {len(df_aysen_2024):,}")

  Procesando chunk 0...
  Procesando chunk 10...
  Procesando chunk 20...
  Procesando chunk 30...
  Procesando chunk 40...
  Procesando chunk 50...
  Procesando chunk 60...
  Procesando chunk 70...
  Procesando chunk 80...
  Procesando chunk 90...
  Procesando chunk 100...
  Procesando chunk 110...
  Procesando chunk 120...
  Procesando chunk 130...
  Procesando chunk 140...
  Procesando chunk 150...
  Procesando chunk 160...
  Procesando chunk 170...
  Procesando chunk 180...

✓ Extracción completa.
  Registros de Aysén encontrados: 100,745


---
## 2. TRANSFORMACIÓN

### 2.1 Limpieza y normalización de columnas

In [4]:
# ── Limpieza de tipos ──────────────────────────────────────────────────────────
# Convertir 'edad' a numérico; valores inválidos se reemplazan por 0
df_aysen_2024['edad'] = (
    pd.to_numeric(df_aysen_2024['edad'], errors='coerce')
    .fillna(0)
    .astype(int)
)

# Asegurar que 'comuna' sea entero para el mapeo
df_aysen_2024['comuna'] = df_aysen_2024['comuna'].astype(int)

# ── Mapeo de etiquetas ─────────────────────────────────────────────────────────
df_aysen_2024['nombre_comuna'] = df_aysen_2024['comuna'].map(COMUNAS)
df_aysen_2024['sexo_label'] = df_aysen_2024['sexo'].map({1: 'Hombre', 2: 'Mujer'})

# ── Verificación de calidad ────────────────────────────────────────────────────
nulos_comuna = df_aysen_2024['nombre_comuna'].isna().sum()
nulos_sexo   = df_aysen_2024['sexo_label'].isna().sum()
print(f"Registros con nombre_comuna no mapeado: {nulos_comuna}")
print(f"Registros con sexo_label no mapeado:    {nulos_sexo}")

df_aysen_2024.head()

Registros con nombre_comuna no mapeado: 0
Registros con sexo_label no mapeado:    0


,region,comuna,sexo,edad,edad_quinquenal,nombre_comuna,sexo_label
0,11,11202,2,-66,30,Cisnes,Mujer
1,11,11202,1,-66,55,Cisnes,Hombre
2,11,11202,1,-66,5,Cisnes,Hombre
3,11,11201,1,60,60,Aysén,Hombre
4,11,11202,1,-66,65,Cisnes,Hombre


### 2.2 Indicador 1 — Distribución por sexo y grupo etario quinquenal

In [5]:
indicador_sexo_edad = (
    df_aysen_2024
    .groupby(['nombre_comuna', 'edad_quinquenal', 'sexo_label'])
    .size()
    .reset_index(name='poblacion')
    .sort_values(['nombre_comuna', 'edad_quinquenal'])
)

print("=== Distribución por sexo y grupo etario (muestra: Chile Chico) ===")
print(indicador_sexo_edad[indicador_sexo_edad['nombre_comuna'] == 'Chile Chico'].to_string(index=False))

=== Distribución por sexo y grupo etario (muestra: Chile Chico) ===
nombre_comuna  edad_quinquenal sexo_label  poblacion
  Chile Chico                0     Hombre        117
  Chile Chico                0      Mujer        109
  Chile Chico                5     Hombre        166
  Chile Chico                5      Mujer        142
  Chile Chico               10     Hombre        162
  Chile Chico               10      Mujer        166
  Chile Chico               15     Hombre        144
  Chile Chico               15      Mujer        144
  Chile Chico               20     Hombre        111
  Chile Chico               20      Mujer         94
  Chile Chico               25     Hombre        150
  Chile Chico               25      Mujer        157
  Chile Chico               30     Hombre        190
  Chile Chico               30      Mujer        206
  Chile Chico               35     Hombre        202
  Chile Chico               35      Mujer        217
  Chile Chico               40 

### 2.3 Indicador 2 — Índice de envejecimiento por comuna

**Fórmula:** `(población 65+) / (población 0–14) × 100`

Se usa `edad_quinquenal` en lugar de `edad` individual por ser la columna más confiable para toda la región (incluye Chile Chico).

In [6]:
mayores = df_aysen_2024[df_aysen_2024['edad_quinquenal'] >= 65].groupby('nombre_comuna').size()
jovenes = df_aysen_2024[df_aysen_2024['edad_quinquenal'] <= 10].groupby('nombre_comuna').size()

indice_envejecimiento = pd.DataFrame({
    'pob_65_mas': mayores,
    'pob_0_14':   jovenes
}).fillna(0)

indice_envejecimiento['indice_envejecimiento'] = (
    (indice_envejecimiento['pob_65_mas'] / indice_envejecimiento['pob_0_14'] * 100)
    .round(1)
)

print("=== Índice de Envejecimiento por Comuna (CENSO 2024) ===")
print(indice_envejecimiento.sort_values('indice_envejecimiento', ascending=False).to_string())

=== Índice de Envejecimiento por Comuna (CENSO 2024) ===
               pob_65_mas  pob_0_14  indice_envejecimiento
nombre_comuna                                             
Lago Verde            159        87                  182.8
Río Ibáñez            509       420                  121.2
Chile Chico           801       862                   92.9
Cochrane              442       680                   65.0
Coyhaique            7192     11220                   64.1
Cisnes                608       949                   64.1
Aysén                2746      4459                   61.6
Guaitecas             160       283                   56.5
O'Higgins              55       132                   41.7
Tortel                 33       107                   30.8


---
## 3. CARGA (Exportación local)

Se exportan los datos procesados a CSV para su posterior carga en PostgreSQL (Sprint 7).

In [7]:
# Datos base transformados
df_aysen_2024.to_csv(RUTA_SALIDA_2024, index=False)
print(f"✓ Datos base guardados en: {RUTA_SALIDA_2024}")

# Indicadores calculados
indicador_sexo_edad.to_csv('data/indicador_sexo_edad.csv', index=False)
indice_envejecimiento.to_csv('data/indice_envejecimiento.csv')
print("✓ Indicadores exportados.")
print("\n[Pendiente S7] Carga en PostgreSQL via SQLAlchemy + psycopg2")

✓ Datos base guardados en: data/censo_2024_aysen_final.csv
✓ Indicadores exportados.

[Pendiente S7] Carga en PostgreSQL via SQLAlchemy + psycopg2
